# AI应用基础 · 工单M3.1《产品表面缺陷尺寸测量系统》 — 教师参考版
第8周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m3"], check=True)
print("数据已生成: data/")

工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础


M3.1 -> data/surface_parts 40 件(ground_truth.json为教师核对用, 勿下发)
完成。数据输出目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础/data
数据已生成: data/


## 参考实现 `m31`（含依赖导入）

In [2]:
import csv
import json
import sys
import numpy as np
import numpy as np  # 通用科学计算


def m31():
    import cv2

    PX = 0.1
    for i in range(3):
        p = f"data/surface_parts/part_{i:03d}.png"
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        blur = cv2.GaussianBlur(img, (5, 5), 0)
        _, bw = cv2.threshold(blur, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN,
                              np.ones((3, 3), np.uint8))
        cnts, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
        anno = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        for c in cnts:
            if cv2.contourArea(c) < 15:
                continue
            rect = cv2.minAreaRect(c)
            w, h = sorted(rect[1])
            if w / max(h, 1) > 3:                      # 划痕
                val = w * PX
                txt = f"scratch {val:.2f}mm"
            else:                                      # 凹坑
                val = cv2.contourArea(c) * PX ** 2
                txt = f"pit {val:.2f}mm2"
            box = cv2.boxPoints(rect).astype(int)
            cv2.drawContours(anno, [box], 0, (0, 0, 255), 2)
            cv2.putText(anno, txt, (box[:, 0].min(), box[:, 1].min() - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)
            print(p.split("/")[-1], txt)
        cv2.imwrite(f"result_{i:03d}.png", anno)


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `1` | TODO-1 预处理: 高斯模糊(5×5) → 阈值分割(先试100, 再试Otsu) → 开运算去噪点 |
|  | （在此处编写你的代码——依据上方注释/文档字符串） |
| `2` | TODO-2 轮廓: cv2.findContours(binary, cv2.RETR_EXTERNAL, ...) |
| `3` | TODO-3 判型: 最小外接矩形长宽比>3→划痕(长度=长边), 否则→凹坑(面积) |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） TODO-4 换算: 长度mm = 长边px * PX_TO_MM |
| `5` | TODO-5 标注: cv2.drawContours + cv2.putText(类型+尺寸) |
| `6` | TODO-6 决策矩阵: 全局100 / Otsu / 自适应 三种阈值各测5件, 记录检出与误检 |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 执行与验证

In [3]:
m31()

part_000.png pit 2.24mm2
part_001.png pit 3.28mm2
part_002.png pit 2.77mm2
